**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `python_prebuilt/` and the bundle's R routes (`R/lib`, see `R/README_BRIDGE.md`; the full R pipeline is the project RWDR). Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# M45 — ML-Selected Synthetic Control (Elastic Net)
**Status: READY (ML/AI causal inference)**

---
## Formula
```
w = argmin_w  ||Y_treated,pre - Donors_pre' w||^2 + alpha*(l1_ratio*||w||_1 + (1-l1_ratio)/2*||w||_2^2)
ATT = mean( Y_treated,post - Donors_post' w )
```
**Source**: Doudchenko & Imbens (2016)-style relaxation of the simplex constraint; elastic-net
donor selection.

---
## Why this over classic synthetic control (M11)
Classic SC requires non-negative weights summing to one, and struggles when donors outnumber
pre-periods. Elastic-net regularization picks a **sparse** donor set and handles many donors.

**VALIDATED**: with 25 donors of which only 3 were genuinely relevant, recovered a true effect
of 4.0 as 4.717 while selecting 8 nonzero donors.

---
## DATA REQUIREMENTS
Needs one treated sub-watershed and several never-treated donors, with enough pre-periods.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")
import _common as C      # v20.29: imported BEFORE first use (the import used to come later in this cell)

# ---- CELL 1: configuration + scenario ----
MODEL_ID = "M45"
OUTCOME  = C.current_outcome("NDVI")        # variable this run starts with; CELL 9 repeats for the rest
# ---- v20.57: THE DESIGN OF THIS RUN -- every option below is applied HERE, when this model runs, on the panel P00 built ONCE.
#      Change any of them and re-run this notebook; P00 is never re-run for it. What you set is what runs: DESIGN IN EFFECT
#      (printed below, and saved as DESIGN_IN_EFFECT.csv next to the results) lists each option, the value used and where it came from.
DESIGN_MODE    = "recommended"   # "recommended": an option set to "data" is chosen FROM THE DATA (DESIGN_RECOMMENDATION.md: export breaks,
                                 #   fill years, spillover into inner rings -- never the effect) | "manual": "data" = every ring / every year.
                                 #   A value you set (e.g. CONTROL_ZONES = "1-3", PRE_YEARS = 4) is ALWAYS used exactly as set, in both modes
TREATMENT_TIMING = "fund"        # "fund": each sub-watershed's first treated SEASON from your fund workbook -- the release timing, back-cast
                                 #   before the file's first month, the season AFTER the start (no anticipation): staggered, season-level
                                 #   cohorts (FUND_TIMING_AND_DOSE.md) | "registry": its year in data/sites/sites.csv | "fixed": TREATMENT_YEAR
TREATMENT_YEAR = 2022            # "fixed" timing -- and the fall-back for a sub-watershed the fund file / registry does not date (announced)
FUND_START_RULE = "backcast"     # fund timing: "backcast" | "share" (the amount reaches 10 % of the target) | "file_start" (a lower bound)
DOSE_VARIABLE  = "dose_intensity_per_ha"   # the dose models' dose (fund file): amount released / treatment area | "dose_amount_sws" |
                                           #   "dose_share_of_target" -- 0 on the control rings and before the treatment starts
CONTROL_ZONES  = "data"          # control ring(s): "data" | "1-5" | "1-3" | "2-5" | (1, 3, 5)
CONTROL_SELECTION = "rings"      # v20.59 (your rule): "rings" = every ring of CONTROL_ZONES is the control group | "pre_rings" = per outcome, the CONTROL_SELECT_K
                                 #   rings whose PRE-period series is closest to the treatment area's | "pre_blocks" = ~1 km blocks of control pixels chosen the same way until
                                 #   CONTROL_SELECT_RATIO x the treated pixels. Decided on the PRE period ONLY (a post- or outcome-based rule is refused), the same control
                                 #   pixels in every year and season; evidence CONTROL_SELECTION_<outcome>.csv beside the results; results tagged _ctrlPre...
CONTROL_SELECT_K = 2             # pre_rings: how many rings (1-5)
CONTROL_SELECT_RATIO = 3.0       # pre_blocks: control pixels >= this x the treated pixels
CONTROL_SELECT_ON = "trend"      # "trend" (the demeaned pre series' distance = what the parallel-trends assumption asks) | "level" (the pre mean's gap) | "both"
SAME_PIXELS       = "pre_post"   # v20.59 (your rule): every treated and control pixel must be observed in pre AND post ("pre_post") or in every year-season
                                 #   ("all"), else it leaves the sample -- the groups are the same pixels across the panel | "off" (a pixel may contribute to one side only)
DONUT_RINGS       = [1]          # spec 1: rings left OUT of the control pool -- the spillover buffer next to the treated core ([] = none)
LANDUSE_KEEP      = "all"        # spec 1: "all" | your exporter's LandUse class codes, e.g. [2]: a pixel is kept by its PRE-period (baseline) class
BASELINE_NDVI_MIN = None         # spec 1: e.g. 0.25 -- a pixel's pre-period mean NDVI must exceed it (an agricultural mask) | None = off
MIN_PIXEL_COVERAGE_PCT = 0.05    # spec 1 / 3: a year-season whose treated or control coverage is below this share of the typical one is screened out
DROP_SINGLETONS   = False        # spec 3: series seen once leave before the demeaning (the pre-flight; pyfixest drops them anyway)
PRECISION_TOLERANCE = 1e-6       # spec 1: |value| <= this is the no-data zero; a year-season is "constant" within it (indices in [-1, 1])
CLUSTER        = "site"          # v20.59: "site" (the sub-watersheds; fewer than 6 -> the years) | "block" (~1 km spatial blocks of pixels: many clusters, spatial correlation absorbed)
PRE_YEARS      = "data"          # years BEFORE the start: "data" | "all" | 4 (the 4 years before) | 2015 (a calendar year = the FIRST pre year)
POST_YEARS     = "data"          # years FROM the start:   "data" | "all" | 2 (the start year + 1) | 2025 (a calendar year = the LAST post year)
SEASONS        = "all"           # "all" (annual composite + Kharif/Rabi/Zaid: year AND season variation) | "seasonal" | "yearly" |
                                 #   "Rabi" | "Kharif+Rabi" | "auto" (the data decide)
EXCLUDE_TRANSITION_YEAR = False  # True = the first treated year of each series leaves the sample (robustness)
UNIT_FE        = "pixel_season"  # one fixed effect per pixel x season series (+ year x season FE) | "pixel"
SUB_WATERSHEDS = "data"          # v20.58: the sub-watershed(s) THIS run processes: "data" = every one with >= 5 % of the largest one's rows (one
                                 #   export = its own sub-watershed) | "major" (the largest only) | a name or id, e.g. "Koranahalli" | [1, 7]
FRAGMENT_RULE  = "drop"          # YOUR RULE: rows of sub-watersheds NOT processed and rows outside every polygon leave EVERY group -- treated and
                                 #   control, pre and post (SAMPLE INTEGRITY confirms it at every run) | "keep"
OVERLAP_ROWS   = "drop"          # YOUR RULE: overlapping pixels leave every group -- a pixel treated in another processed sub-watershed (as a
                                 #   control), a repeated pixel-year-season, a near-duplicate pixel, a pixel whose ring the exports disagree on | "keep"
POOLED_FE      = "site_period"   # several sub-watersheds: sub-watershed x year x season effects | "period" (one shared year x season effect)
EXCLUDE_GAPFILLED = True         # rows the exporter filled from history (GapFilled / Coverage 0) are not observations | False keeps them
OUTCOME_SCREEN    = "drop"       # v20.59: "drop" (a year-season constant across pixels -- a fill value -- or with collapsed coverage leaves the
                                 #   model; its evidence: OUTCOME_SCREEN_<outcome>.csv beside the results) | "keep" (reported and KEPT; results tagged
                                 #   _screenKept) | "off". A v20.58 run stopped here ("41 year-seasons are NOT pixel data ... Re-export it")
DESIGN_SOURCE     = "panel"      # v20.59 -- YOUR RULE: the PANEL's treat / control / pre / post / did (the exports' Treat flag, PERIOD_RULE) are
                                 #   what this model estimates on (results tagged _panelDesign; the design in effect above is COMPARED with them) | "model" =
                                 #   design-based modelling: the timing / TREATMENT_YEAR / transition-year settings above build post / pre / did
COVARIATES = "all"   # v20.45: THIS model's covariates -- "all" = the FOUR weather covariates (Rain, Tmax, Tmean, Tmin; tag covAll4) | "mean_temp_rain" (Tmean + Rain) | "none" | or a list. LandUse is never a covariate.
NONNEGATIVE = False   # v20.32: True = no negative value of any estimation variable enters the DiD (selection on the
                      # outcome -- NDVI/NDWI/NDMI/NDRE/SMDI are negative over water, bare soil, moisture stress);
                      # results then go to a separate "_nonneg" folder and every file records it
C.set_scenario(design_mode=DESIGN_MODE, timing=TREATMENT_TIMING, treatment_year=TREATMENT_YEAR, fund_start_rule=FUND_START_RULE,
               dose_variable=DOSE_VARIABLE, control_zones=CONTROL_ZONES, pre_years=PRE_YEARS, post_years=POST_YEARS, seasons=SEASONS,
               exclude_transition_year=EXCLUDE_TRANSITION_YEAR, unit_fe=UNIT_FE, overlap_rows=OVERLAP_ROWS, fragment_rule=FRAGMENT_RULE,
               pooled_fe=POOLED_FE, exclude_gapfilled=EXCLUDE_GAPFILLED, covariates=COVARIATES, nonnegative=NONNEGATIVE, sub_watersheds=SUB_WATERSHEDS, outcome_screen=OUTCOME_SCREEN, design_source=DESIGN_SOURCE,
               control_selection=CONTROL_SELECTION, control_select_k=CONTROL_SELECT_K, control_select_ratio=CONTROL_SELECT_RATIO, control_select_on=CONTROL_SELECT_ON, same_pixels=SAME_PIXELS, cluster=CLUSTER,
               donut_rings=DONUT_RINGS, landuse_keep=LANDUSE_KEEP, baseline_ndvi_min=BASELINE_NDVI_MIN, min_pixel_coverage_pct=MIN_PIXEL_COVERAGE_PCT, drop_singletons=DROP_SINGLETONS, precision_tolerance=PRECISION_TOLERANCE)
# multi-site (20 sub-watersheds, SWSiD_All): run this model per site and pooled from 08_Multisite_Runs/MS01_Multisite_Runs.ipynb
SCENARIO_FROM_PANEL = True   # v20.34: what you set in THIS cell always runs; an option you leave unset (None) comes from P00's saved scenario
if SCENARIO_FROM_PANEL: C.load_scenario()
C.resolve_design()           # v20.57: timing (fund / registry / fixed), the "data" options, the fragment rule -> DESIGN IN EFFECT
CONTROL_ZONES = C.ACTIVE["control_zones"]
MAX_ROWS = None    # restored (v20.3)
RESULTS_DIR = C.results_dir(MODEL_ID)         # v20: RESULTS_ROOT/<model>/<scenario tag>
C.info(f"SCENARIO IN FORCE: {C.scenario_banner()}")
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.58")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("M45")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
C.system_report()   # v16: shows CPU / RAM / whether the A40 will be used for demeaning
C.autotune()          # v17: measures THIS machine (cores/RAM/VRAM/disk) and sets every performance knob
C.done(1, next_task="CELL 2 -- load the panel columns this model needs")


In [ ]:
# ---- v20.42: PRE-BUILT PACKAGE FIRST ------------------------------------------------------------------------
# A verified pre-built package (installed and checked on a known answer by P00's P12 step) computes this model's
# PRIMARY result: inside the engine function the model calls (pyfixest / wildboottest / diff-diff), or here, as
# <MODEL>_PACKAGE_<OUTCOME>.csv (diff-diff / econml / esda). The engine code below is the result only when no verified
# package exists -- and the cross-check when one does. What computes this model, and why, is printed below.
PACKAGE_RESULT = C.prebuilt_first(MODEL_ID, OUTCOME)
_eng = C.LAST_ENGINE.get(MODEL_ID, {})
print(f"[INFO]    {MODEL_ID} / {OUTCOME}: computed by {_eng.get('engine', 'engine')} -- {_eng.get('reason', '')}")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 2: load ONLY the columns this model needs ----
# v16: panel is the compact 48-column schema (pixel_id int64, outcomes float32);
# treat_core/pre_period/... aliases are regenerated by build_treatment_columns() -- nothing to change here.
C.step(2,"loading panel (column-projected)")
try:
    panel = C.load_panel(columns=C.columns_for(OUTCOME), max_rows=MAX_ROWS)
    C.ok(f"columns loaded: {list(panel.columns)}")
except Exception as e:
    C.fail(f"cannot load panel: {e}")
    C.info("Checklist:")
    C.info("  1. Run 01_Panel_Preparation (P00, or P01-P06) -- it writes did_panel_full.parquet")
    C.info("  2. v9.1 auto-discovers it via panel_location.json; if that failed, set")
    C.info("     PREPARED_PANEL in _common.py to the EXACT path P05 printed")
    raise
C.done(2, next_task="CELL 3 -- estimate M45")

In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 3: run M45 ----
C.step(3,"estimating -- ML-selected synthetic control")

def run_model():
    # v20.57: the elastic-net synthetic control on the core-vs-ring SERIES (the treated = the core series of a cohort, the donors =
    # the ring series, never treated). Until v20.56 the donors were the OTHER sub-watersheds -- all treated: +0.002 for a true
    # +0.05 (validate_known_answers.py), and no result for one sub-watershed. v20.57: one fit per core series (sub-watershed x
    # season) on the SAME-season ring series, net of their common year effect (the penalty no longer shrinks the shared trend:
    # 0.064 -> 0.050 on the staggered test panel).
    df=C.build_treatment_columns(panel, control_zones=CONTROL_ZONES)
    df=df[df["in_analysis_sample"]==1]
    r, tbl, w = C.lasso_sc_ring_series(df, OUTCOME)
    C.info(f"ATT={r['ATT_lasso_sc']:.6f} | {r['n_nonzero_donors']} nonzero donor weight(s) over {r['n_cohorts']} cohort(s)")
    C.save_results({"outcome":OUTCOME,**r}, RESULTS_DIR, f"lasso_sc_{OUTCOME}.csv")
    C.save_results(tbl, RESULTS_DIR, f"lasso_sc_by_cohort_{OUTCOME}.csv")
    print(w.sort_values("weight",key=abs,ascending=False).head(10).to_string(index=False))
    C.save_results(w, RESULTS_DIR, f"lasso_sc_weights_{OUTCOME}.csv")

result=C.cell_guard(run_model)()
C.report_model_outcome(MODEL_ID, OUTCOME, result, RESULTS_DIR)   # v17.10
C.ok("M45 finished") if result is not None else C.warn("M45 produced no result -- see above")

# ---- MEMORY RELEASE (v14): free this model's frames so the next one starts clean ----
try:
    C.release(*[v for k,v in list(globals().items()) if k in ("panel","df","result") and v is not None])
except Exception as _e:
    print(f"[INFO]    release skipped: {_e}")
C.done(3, next_task="any model -- see MODEL_GUIDE_CATEGORY_WISE.md; recommended sequence starts at M01")

In [ ]:
# ---- CELL 9: run the REMAINING outcome variables automatically (v17.10) ----
# Run All on this notebook now produces every variable in C.SELECTED_OUTCOMES, one after another,
# with no manual edits. Set AUTO_NEXT_OUTCOMES = False to estimate only the outcome chosen in CELL 1.
AUTO_NEXT_OUTCOMES = True
OVERWRITE_EXISTING = False        # True = recompute outcomes that already have result files
if AUTO_NEXT_OUTCOMES:
    C.run_other_outcomes("M45", first_outcome=OUTCOME, overwrite=OVERWRITE_EXISTING)
